In [3]:
# !pip install jupyter-ui-poll

In [1]:
import os
from IPython.display import display, Image, HTML, Javascript, clear_output
import ipywidgets as widgets
from PIL import Image
import numpy as np
import io
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from jupyter_ui_poll import ui_events
import time
import re
import requests
import warnings
from io import BytesIO
from astropy.visualization import ZScaleInterval
from astropy.io import fits
from astropy.wcs import WCS
import threading

In [57]:

# csv_path_and_name='/global/cfs/projectdirs/cosmo/work/users/xhuang/xh-repos/DR10-lens-search/VI/XH-VI_2025May29_All-combined.csv'
# csv_path_and_name='/global/cfs/cdirs/cosmo/work/users/xhuang/xh-repos/DR10-lens-search/VI/dimples-for-vi-HST-18085.csv'
# csv_path_and_name='/global/cfs/cdirs/cosmo/work/users/xhuang/xh-repos/DR10-lens-search/VI/dimples_2025Aug1-As2.csv'
# csv_path_and_name='/global/homes/j/jreyes/HST/Data/DR2_knowns.csv'
# csv_path_and_name='/global/cfs/projectdirs/cosmo/work/users/xhuang/xh-repos/DR10-lens-search/VI/dimple-targets/2025Aug6-dimple-selections.csv'
csv_path_and_name = 'dimple_with_vac.csv'

df = pd.read_csv(csv_path_and_name)

# Convert 'xh_grade' to string (handles NaNs as empty strings)
try:
    df['xh_Grade'] = df['xh_Grade'].fillna('').astype(str)
except Exception as e:
    print("Failed to convert 'xh_Grade' to string or save CSV. Probably the Grader is not xh")
    print("Exception details:", repr(e))


df.reset_index(drop = True, inplace = True)

csv_idx = 2
current_index = csv_idx - 2

name_input = widgets.Text(
    value='',
    placeholder='Enter your name',
    description='Name:',
    disabled=False
)

enter_button = widgets.Button(
    description='Enter',
    disabled=False,
    button_style='',
    tooltip='Click to enter',
    icon='check'
)

input_widgets = widgets.HBox([name_input, enter_button])

button_clicked = False

def on_click(b):
    global button_clicked
    button_clicked = True
    global commenter_name
    commenter_name = name_input.value

enter_button.on_click(on_click)

display(input_widgets)

with ui_events() as poll:
    while button_clicked is False:
        poll()     #checking if name has been input
        time.sleep(0.1)

clear_output(wait=True) # wait for images to be ready, then clear name box 

if (f'{commenter_name}_Grade' not in list(df.columns)) and (f'{commenter_name}_Comments' not in list(df.columns)):
    df[f'{commenter_name}_Grade']=None
    df[f'{commenter_name}_Comments']=None

# Widgets
image_size='200px'
image_widget = widgets.Image(layout=widgets.Layout(width=image_size, height=image_size))
image_widget_g = widgets.Image(layout=widgets.Layout(width=image_size, height=image_size))
image_widget_r = widgets.Image(layout=widgets.Layout(width=image_size, height=image_size))
image_widget_z = widgets.Image(layout=widgets.Layout(width=image_size, height=image_size))

title_widget = widgets.HTML(layout=widgets.Layout(text_align='center'))
predictions_title_widget = widgets.HTML(layout=widgets.Layout(text_align='center'))
link_widget = widgets.HTML(layout=widgets.Layout(text_align='center'))
text_widget = widgets.Text(description="Grade:")
text_widget_comment = widgets.Text(description="Comment:")
index_widget_comment = widgets.Text(description="Skip to idx:") #NEW
url_display = widgets.HTML(value='')

# code that causes 'ghost comment' -- can be deleted 8/6/2025
# if pd.isna(df.loc[0,f'{commenter_name}_Grade']):
#     text_widget.value = ''
# else:
#     text_widget.value = df.loc[0,f'{commenter_name}_Grade']

# if pd.isna(df.loc[0,f'{commenter_name}_Comments']):
#     text_widget_comment.value = ''
# else:
#     text_widget_comment.value = df.loc[0,f'{commenter_name}_Comments']


if pd.isna(df.loc[current_index,f'{commenter_name}_Grade']):
    text_widget.value = ''
else:
    text_widget.value = df.loc[current_index,f'{commenter_name}_Grade']

if pd.isna(df.loc[current_index,f'{commenter_name}_Comments']):
    text_widget_comment.value = ''
else:
    text_widget_comment.value = df.loc[current_index,f'{commenter_name}_Comments']

button_prev = widgets.Button(description="Previous")
button_next = widgets.Button(description="Next")
button_skip = widgets.Button(description="Skip to index") #NEW
button_sky_viewer = widgets.Button(description="Show on Sky Viewer")

output = widgets.Output()

def sdss_rgb(imgs, bands = ['g','r','z'], scales=None, m=0.03, Q=20, mnmx=None):
    imgs=imgs.swapaxes(0,2)
    rgbscales=dict(g=(2, 6.0),
                   r=(1, 3.4),
                   i=(0, 3.0),
                   z=(0, 2.2))
    if scales is not None:
        rgbscales.update(scales)
    I = 0
    for img,band in zip(imgs, bands):
        plane,scale = rgbscales[band]
        img = np.maximum(0, img * scale + m)
        I = I + img
    I /= len(bands)
    if Q is not None:
        fI = np.arcsinh(Q * I) / np.sqrt(Q)
        I += (I == 0.) * 1e-6
        I = fI / I
    H,W = I.shape
    rgb = np.zeros((H,W,3), np.float32)
    for img,band in zip(imgs, bands):
        plane,scale = rgbscales[band]
        if mnmx is None:
            rgb[:,:,plane] = np.clip((img * scale + m) * I, 0, 1)
        else:
            mn,mx = mnmx
            rgb[:,:,plane] = np.clip(((img * scale + m) - mn) / (mx - mn), 0, 1)
    return rgb

def get_cutout(brickname,objid):
    path='/global/cfs/projectdirs/cosmo/work/users/xhuang/dr10_1/south/z_lt20/coadd/'+brickname[:3]+'/'+str(brickname)+'.h5'
    cutout = h5py.File(path, 'r')['o'+str(int(objid))]
    return np.array(cutout,dtype=np.float32)

def get_cutout_by_radec(ra, dec, cutout_size=101):
    ''' YMH+JI's addition to get cutout w/o target id -- 7/29/25 '''

    url = 'https://www.legacysurvey.org/viewer-desi/cutout.jpg?ra={}&dec={}&layer={}&pixscale={}&size={}&zoom=16'.format(ra, dec, 'ls-dr10-grz', 0.262, int(cutout_size))
    # response = requests.get(url)
    response = requests.get(url, auth=('desi', '[REDACTED]'))

    if response.status_code == 200:
        image_buffer = BytesIO(response.content)
        image = Image.open(image_buffer)
        return np.array(image)
    else:
        raise Exception(f"Failed to fetch image. Status code: {response.status_code}")

def get_cutout_by_radec_hsc(ra, dec, cutout_size=147):
    '''CS's addition to get (grz) fits cutout -- 8/04/25 '''

    url = 'https://www.legacysurvey.org/viewer/cutout.jpg?ra={}&dec={}&layer={}&pixscale={}&size={}&zoom=16'.format(ra, dec, 'hsc-dr3', 0.18, int(cutout_size))
    response = requests.get(url)
    size = int(response.headers.get("Content-Length", 0))
    if size>14000:    # ensures more than one HSC band   
        image_buffer = BytesIO(response.content)
        image = Image.open(image_buffer)
        return (np.array(image),True)
    else:
        return (get_cutout_by_radec(ra,dec),False)
    

def get_fits_cutout_by_radec(ra, dec, cutout_size=101):
    '''CS's addition to get (grz) fits cutout -- 8/01/25 '''

    url = 'https://www.legacysurvey.org/viewer/cutout.fits?ra={}&dec={}&layer={}&pixscale={}&size={}&zoom=16'.format(ra, dec, 'ls-dr10-grz', 0.262, int(cutout_size))
    return(fits.open(url)[0].data)

def get_fits_cutout_by_radec_hsc(ra, dec, cutout_size=147):
    '''CS's addition to get (grz) fits cutout -- 8/04/25 '''

    url = 'https://www.legacysurvey.org/viewer/cutout.fits?ra={}&dec={}&layer={}&pixscale={}&size={}&zoom=16'.format(ra, dec, 'hsc-dr3', 0.18, int(cutout_size))
    return(fits.open(url)[0].data)

def get_xy(center_ra, center_dec, object_ra, object_dec, size_pixels=101, pixscale=0.262):
    w = WCS(naxis=2)
    w.wcs.crpix = [size_pixels / 2, size_pixels / 2]  # reference pixel (center)
    w.wcs.cdelt = np.array([-pixscale / 3600, pixscale / 3600])  # deg/pixel
    w.wcs.crval = [center_ra, center_dec]  # reference coordinates
    w.wcs.ctype = ["RA---TAN", "DEC--TAN"]  # projection type
    w.wcs_world2pix([[3.7379, 1.6170]], 0)[0]

    x, y = w.wcs_world2pix([[object_ra, object_dec]], 0)[0]
    return x, y
    
def plot_cutout_with_redshifts(ax,rgb_cutout,ra,dec,hsc,redshifts_df):
    if hsc:
        ax.imshow(np.flipud(rgb_cutout),origin='lower')
        if redshifts_df is not None:
            y_coords_for_zs = [20*(n+1) for n in range(len(redshifts_df))]
            
            for i in range(len(redshifts_df)):
                x,y = get_xy(ra, dec, object_ra=redshifts_df['ra'].values[i], object_dec=redshifts_df['dec'].values[i], size_pixels=147, pixscale=0.18)
                line, =ax.plot([x,100],[y,y_coords_for_zs[i]],ls='--')
                ax.text(100,y_coords_for_zs[i],redshifts_df['z'].values[i],va='center',ha='left',size=15,color=line.get_color())
        ax.axis('off')
        ax.set_title('RGB with redshifts')
    else:
        ax.imshow(np.flipud(rgb_cutout),origin='lower')
        if redshifts_df is not None:
            y_coords_for_zs = [20*(n+1) for n in range(len(redshifts_df))]
            
            for i in range(len(redshifts_df)):
                x,y = get_xy(ra, dec, object_ra=redshifts_df['ra'].values[i], object_dec=redshifts_df['dec'].values[i], size_pixels=101, pixscale=0.262)
                line, =ax.plot([x,80],[y,y_coords_for_zs[i]],ls='--')
                ax.text(80,y_coords_for_zs[i],redshifts_df['z'].values[i],va='center',ha='left',size=15,color=line.get_color())
        ax.axis('off')
        ax.set_title('RGB with redshifts')
    
slider_g = widgets.FloatSlider(value=8, min=0.1, max=20, step=0.1, description='G vmax:', continuous_update=True)
slider_r = widgets.FloatSlider(value=8, min=0.1, max=20, step=0.1, description='R vmax:', continuous_update=True)
slider_z = widgets.FloatSlider(value=8, min=0.1, max=20, step=0.1, description='Z vmax:', continuous_update=True)

slider_box = widgets.HBox([slider_g, slider_r, slider_z])
display(slider_box)


class ImageData:
    def __init__(self, df):
        self.df = df
        self.saved_fits={}
        self.saved_npy={}
        self.saved_redshifts={}
    
    def update_image(self,i):
        self.current_index=i
        self.stored_idxs = list(self.saved_fits.keys())

        if self.current_index not in self.stored_idxs:
            try:
                ra, dec = self.df['ra'].iloc[self.current_index], self.df['dec'].iloc[self.current_index]
            except:
                ra, dec = self.df['Ra'].iloc[self.current_index], self.df['Dec'].iloc[self.current_index]

            npy_array,hsc = get_cutout_by_radec_hsc(ra, dec)

            if hsc:
                # Get z with cutout size 147
                z_df = self.get_redshifts(ra,dec,cutout_size=147)
                fits_array = get_fits_cutout_by_radec_hsc(ra, dec)
            else:
                # Get z with cutout size 101
                z_df = self.get_redshifts(ra,dec,cutout_size=101)
                fits_array = get_fits_cutout_by_radec(ra, dec)

            self.saved_fits[self.current_index] = np.copy(fits_array)
            self.saved_npy[self.current_index] = (np.copy(npy_array), hsc)
            self.saved_redshifts[self.current_index] = z_df.copy()

        neighbor_idxs = self.generate_neighbor_idxs()
        
        for neighbor in neighbor_idxs:
            t = threading.Thread(target=self.prepare_neighbor_data, args=(neighbor,), daemon=True)
            t.start()
            
        return self.saved_fits[self.current_index], self.saved_npy[self.current_index][0], self.saved_npy[self.current_index][1], self.saved_redshifts[self.current_index]

    def generate_neighbor_idxs(self):
        if self.current_index == 0:
            neighbor_idxs = [1]
        elif self.current_index == len(df)-1:
            neighbor_idxs = [self.current_index-1]
        else:
            neighbor_idxs=[self.current_index-1,self.current_index+1]
        return neighbor_idxs
        
    def prepare_neighbor_data(self,neighbor_i):
        self.lock = threading.Lock()  # Thread-safe cache
        with self.lock:
            stored_idxs = list(self.saved_fits.keys())

        # If neighbor data aready exists
        if neighbor_i in stored_idxs:
            return
        else:
            try:
                ra, dec = self.df['ra'].iloc[neighbor_i], self.df['dec'].iloc[neighbor_i]
            except:
                ra, dec = self.df['Ra'].iloc[neighbor_i], self.df['Dec'].iloc[neighbor_i]

            npy_array,hsc = get_cutout_by_radec_hsc(ra, dec)

            if hsc:
                # Get z with cutout size 147
                z_df = self.get_redshifts(ra,dec,cutout_size=147)
                fits_array = get_fits_cutout_by_radec_hsc(ra, dec)
            else:
                # Get z with cutout size 101
                z_df = self.get_redshifts(ra,dec,cutout_size=101)
                fits_array = get_fits_cutout_by_radec(ra, dec)

            with self.lock: 
                self.saved_fits[neighbor_i] = np.copy(fits_array)
                self.saved_npy[neighbor_i] = (np.copy(npy_array), hsc)
                self.saved_redshifts[neighbor_i] = z_df.copy()

                # if neighbor_i-1 in stored_idxs:
                #     del self.saved_fits[neighbor_i-1]
                #     del self.saved_npy[neighbor_i-1]
                #     del self.saved_redshifts[neighbor_i-1]
                # if neighbor_i+1 in stored_idxs:
                #     del self.saved_fits[neighbor_i+1]
                #     del self.saved_npy[neighbor_i+1]
                #     del self.saved_redshifts[neighbor_i+1]

    
    def get_redshifts(self,ra,dec,cutout_size):
        diff = 0.262*cutout_size/2/3600
        cat_url = "https://www.legacysurvey.org/viewer-desi/desi-spec-daily/1/cat.json?ralo={}&rahi={}&declo={}&dechi={}".format(ra-diff,ra+diff,dec-diff,dec+diff)
        cat_dict = eval(requests.get(cat_url, auth=('desi', '[REDACTED]')).content)
        
        if len(cat_dict) == 1:
            return None
        else:
            coordinates = np.array(cat_dict['rd'])
            ras, decs = coordinates[:,0],coordinates[:,1]
            redshifts = []
            for item in cat_dict['name']:
                if 'ZWARN' in item:
                    redshifts.append(item.split(' ')[-2]+'\nZWARN')
                elif 'STAR' in item:
                    redshifts.append('STAR')
                else:
                    # print(item)
                    redshifts.append(str(round(float(item.split(' ')[-1]),3)))
            return pd.DataFrame({'ra':ras,'dec':decs,'z':redshifts})
    

ImageDataHolder = ImageData(df)

def update_image(index):
    with output:
        output.clear_output(wait=True)
        try:
            ra, dec = df['ra'].iloc[index], df['dec'].iloc[index]
        except:
            ra, dec = df['Ra'].iloc[index], df['Dec'].iloc[index]
        
        fits_array, npy_array, hsc, redshifts_df = ImageDataHolder.update_image(index)

        fig, axs = plt.subplots(2, 5, figsize=(25, 10))  # 2 rows

        # --- Top row ---
        axs[0, 0].imshow(np.flipud(npy_array), origin='lower')
        if hsc:
            axs[0, 0].set_title('HSC RGB Image')
        else:
            axs[0, 0].set_title('RGB Image')
        axs[0, 0].axis('off')

        plot_cutout_with_redshifts(ax=axs[0, 1], rgb_cutout=npy_array, ra=ra, dec=dec, hsc=hsc,redshifts_df=redshifts_df)

        if hsc:
            cx,cy = 74,74

        else:
            cx,cy = 51,51
            

        # G band with circle
        zscale = ZScaleInterval()
        vmin_g, vmax_g = zscale.get_limits(fits_array[0])
        axs[0, 2].imshow(fits_array[0], origin='lower', cmap='gray', vmin=vmin_g, vmax=vmax_g * slider_g.value)
        axs[0, 2].set_title('G Band')
        axs[0, 2].axis('off')
        axs[0, 2].scatter(cx, cy, marker='o', facecolors='none', edgecolor='red')

        # R band with circle
        zscale = ZScaleInterval()
        vmin_r, vmax_r = zscale.get_limits(fits_array[1])
        axs[0, 3].imshow(fits_array[1], origin='lower', cmap='gray', vmin=vmin_r, vmax=vmax_r * slider_r.value)
        axs[0, 3].set_title('R Band')
        axs[0, 3].axis('off')
        axs[0, 3].scatter(cx, cy, marker='o', facecolors='none', edgecolor='red')

        # Z band with circle
        zscale = ZScaleInterval()
        vmin_z, vmax_z = zscale.get_limits(fits_array[2])
        axs[0, 4].imshow(fits_array[2], origin='lower', cmap='gray', vmin=vmin_z, vmax=vmax_z * slider_z.value)
        axs[0, 4].set_title('Z Band')
        axs[0, 4].axis('off')
        axs[0, 4].scatter(cx, cy, marker='o', facecolors='none', edgecolor='red')

        # --- Bottom row: G, R, Z only, NO circle ---
        for i, (band, vmin, vmax, slider, title) in enumerate(
            zip([fits_array[0], fits_array[1], fits_array[2]],
                [vmin_g, vmin_r, vmin_z],
                [vmax_g, vmax_r, vmax_z],
                [slider_g, slider_r, slider_z],
                ['G Band (no circle)', 'R Band (no circle)', 'Z Band (no circle)'])
        ):
            ax = axs[1, i+2]  # columns 2,3,4
            ax.imshow(band, origin='lower', cmap='gray', vmin=vmin, vmax=vmax * slider.value)
            ax.set_title(title)
            ax.axis('off')

        # Leave the first two bottom-row axes empty for alignment
        axs[1, 0].axis('off')
        axs[1, 1].axis('off')

        plt.tight_layout()
        plt.show()

        update_title(index)
        update_predictions(index)
        url = 'https://www.legacysurvey.org/viewer-desi/?ra={}&dec={}&layer=ls-dr10-grz&pixscale=0.26&size=101&zoom=16'.format(ra, dec)
        url_display.value = f'<a href="{url}" target="_blank">Open on Sky Viewer</a>'

def update_title(index):
    title_widget.value = f"<h3>Index: {current_index + 1} out of {len(df) - 2} (csv index: {current_index + 2}) </h3>"

def update_predictions(index):
    inDGC = 'True' if df.loc[index,'inDGC'] else 'FALSE'
    predictions_title_widget.value="<h4>LOGMSTAR: {}<br>SFR: {}<br>inDGC: {}</h4>".format(df.loc[index, 'LOGMSTAR'],df.loc[index,'SFR'],inDGC)
    
    # predictions_title_widget.value="<h4>ResNet: {:.5f}<br>EfficientNet: {:.5f}</h4>".format(df.loc[index, 'shield_probability'],df.loc[index,'eff_probability'])

# Function to handle button clicks
def on_button_click(b):

    global current_index
    global csv_path_and_name

    # Save current grade
    df.loc[current_index,f'{commenter_name}_Grade']=text_widget.value
    df.loc[current_index,f'{commenter_name}_Comments']=text_widget_comment.value

    # Update index
    if b.description == "Next":
        if current_index < len(df) - 1:
            current_index += 1

    elif b.description == "Previous":
        if current_index > 0:
            current_index -= 1

    elif b.description == "Skip to index": #NEW
        try:
            if int(index_widget_comment.value) >= 1:
                current_index = int(index_widget_comment.value) - 1 #NEW
            index_widget_comment.value = ''
        except:
            index_widget_comment.value = ''

    # Update cutout, title and text
    update_image(current_index)
    if pd.isna(df.loc[current_index,f'{commenter_name}_Grade']):
        text_widget.value = ''
    else:
        text_widget.value = df.loc[current_index,f'{commenter_name}_Grade']
    if pd.isna(df.loc[current_index,f'{commenter_name}_Comments']):
        text_widget_comment.value = ''
    else:
        text_widget_comment.value = df.loc[current_index,f'{commenter_name}_Comments']
        
    # df.to_csv(csv_path_and_name) # uncomment to save during VI


# Assign button click handlers
button_prev.on_click(on_button_click)
button_next.on_click(on_button_click)
button_skip.on_click(on_button_click) 

# Display the interface
centered_box = widgets.Box([title_widget,predictions_title_widget,output, text_widget, text_widget_comment,widgets.HBox([button_prev, button_next]),index_widget_comment,button_skip,url_display], 
                           layout=widgets.Layout(
                               display='flex',
                               flex_flow='column',
                               align_items='center',
                               width='100%'))

display(centered_box)


# --- Make sliders re-render the plot when changed ---
def on_slider_change(change):
    update_image(current_index)

slider_g.observe(on_slider_change, names='value')
slider_r.observe(on_slider_change, names='value')
slider_z.observe(on_slider_change, names='value')

# --- Now update_image(current_index) will work with sliders! ---
update_image(current_index)


# Show the first cutout
# update_image(current_index)

# update_image(100)

Box(children=(HTML(value=''), HTML(value=''), Output(), Text(value='', description='Grade:'), Text(value='', d…

In [21]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
filename = f'2025Aug6-dimple-selections-{timestamp}.csv'
df.to_csv(filename, index=False)

In [13]:
df.to_csv('2025Aug6-dimple-selections.csv', index=False)